# Klasifikasi DemogPairs Menggunakan ViT (Umur) & Logistic Regression

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-age.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [LogisticRegression(random_state=42)],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__max_iter': [500, 1000],
        'classifier__solver': ['lbfgs', 'saga'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

LogisticRegression: 96 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_lr_vit-age_",
    results_path="results/demogpairs_lr_vit-age_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: LogisticRegression


{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}


Accuracy  : 0.8648148148148148
Precision : 0.8648539175117439
Recall    : 0.8648148148148148
F1 Score  : 0.8648007975066698
               precision    recall  f1-score   support

Asian_Females     0.8388    0.8528    0.8457       360
  Asian_Males     0.8430    0.8500    0.8465       360
Black_Females     0.8390    0.8250    0.8319       360
  Black_Males     0.8964    0.8889    0.8926       360
White_Females     0.8764    0.8667    0.8715       360
  White_Males     0.8956    0.9056    0.9006       360

     accuracy                         0.8648      2160
    macro avg     0.8649    0.8648    0.8648      2160
 weighted avg     0.8649    0.8648    0.8648      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9481481481481482,0.8387978142076503,0.8527777777777777,0.8457300275482094,360
Asian_Males,0.9486111111111111,0.8429752066115702,0.85,0.846473029045643,360
Black_Females,0.9444444444444444,0.8389830508474576,0.825,0.8319327731092436,360
Black_Males,0.9643518518518519,0.896358543417367,0.8888888888888888,0.8926080892608089,360
White_Females,0.9574074074074074,0.8764044943820225,0.8666666666666667,0.8715083798882682,360
White_Males,0.9666666666666667,0.8956043956043956,0.9055555555555556,0.9005524861878453,360


Confusion matrix saved: images\cm_lr_vit-age_LogisticRegression.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               307                19                21                 0                12                 1
         Asian_Males                24               306                 5                12                 2                11
       Black_Females                21                 3               297                10                24                 5
         Black_Males                 1                19                12               320                 0                 8
       White_Females                13                 2                19                 1               312                13
         White_Males                 0                14                 0                14                 6               326


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
LogisticRegression,models/clf_demogpairs_lr_vit-age_LogisticRegression.pkl,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.8648148148148148,0.8648007975066698,0.8648539175117439,0.8648148148148148,270


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_lr_vit-age_LogisticRegression.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 3213.0,
 'days': 0,
 'hours': 0,
 'minutes': 53,
 'seconds': 33.0,
 'text': '0 hari 0 jam 53 menit 33.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 36865.0,
 'days': 0,
 'hours': 10,
 'minutes': 14,
 'seconds': 25.0,
 'text': '0 hari 10 jam 14 menit 25.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.849,0.8438,0.8455,0.8588,0.8484,0.8491,0.849,0.8497,0.8491,9.6016
2,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.849,0.8438,0.8455,0.8588,0.8484,0.8491,0.849,0.8497,0.8491,8.8954
3,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 2000, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.849,0.8438,0.8455,0.8588,0.8484,0.8491,0.849,0.8497,0.8491,10.002
4,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 1000, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': None}",0.8495,0.8426,0.8443,0.8571,0.849,0.8485,0.8484,0.8492,0.8485,10.1453
...,...,...,...,...,...,...,...,...,...,...,...
267,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'newton-cg', 'pca': 'PCA', 'scaler': None}",0.64,0.6163,0.6198,0.6181,0.6487,0.6286,0.6252,0.6274,0.6286,3.183
268,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': None}",0.64,0.6163,0.6198,0.6175,0.6487,0.6285,0.6251,0.6272,0.6285,2.7064
269,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 2000, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': None}",0.64,0.6163,0.6198,0.6175,0.6487,0.6285,0.6251,0.6272,0.6285,2.5606
270,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': None}",0.64,0.6163,0.6198,0.6175,0.6487,0.6285,0.6251,0.6272,0.6285,1.925
